In [2]:

!pip install -q scikit-learn ipywidgets

import random
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML
from sklearn.neural_network import MLPClassifier

# Winning combinations
win_lines = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6)
]

# Find a winning move
def find_move(board, mark):
    for line in win_lines:
        values = [board[i] for i in line]
        if values.count(mark) == 2 and values.count("") == 1:
            return line[values.index("")]
    return None

# Prepare training data
X = []
y = []

def encode(board):
    return [1 if x == "X" else -1 if x == "O" else 0 for x in board]

for _ in range(5000):
    board = [
        random.choice(["", "", "", "X", "O"])
        for _ in range(9)
    ]

    if all(board):
        continue

    move = find_move(board, "O")

    if move is None:
        move = find_move(board, "X")

    if move is None:
        empty = [i for i in range(9) if board[i] == ""]
        move = random.choice(empty)

    X.append(encode(board))
    y.append(move)

# Create and train the ANN
ann = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    max_iter=500,
    random_state=42
)

ann.fit(np.array(X), np.array(y))

print("ANN trained successfully!")

# Game variables
board = [""] * 9
player_score = 0
computer_score = 0
draw_score = 0
game_over = False

# Check game result
def check_winner():
    for a, b, c in win_lines:
        if board[a] and board[a] == board[b] == board[c]:
            return board[a]

    if all(board):
        return "Draw"

    return None

# Computer move using ANN
def computer_move():
    empty = [i for i in range(9) if board[i] == ""]
    prediction = ann.predict(np.array([encode(board)]))[0]

    if prediction in empty:
        return int(prediction)

    return random.choice(empty)

# Game heading
display(HTML("""
<div style="background:linear-gradient(135deg,#111827,#581c87);
padding:22px;border-radius:18px;text-align:center;color:white">
<h4 style="color:#c4b5fd;letter-spacing:4px">NEON ARCADE</h4>
<h1>TIC-TAC-TOE</h1>
<p>You are ✕ &nbsp; | &nbsp; ANN Computer is ○</p>
</div>
"""))

status = widgets.HTML(
    value="<h3 style='text-align:center;color:#7c3aed'>Your turn!</h3>"
)

score = widgets.HTML()

buttons = []

for i in range(9):
    button = widgets.Button(
        description=" ",
        layout=widgets.Layout(width="85px", height="75px"),
        style={"font_weight": "bold", "button_color": "#ddd6fe"}
    )
    buttons.append(button)

grid = widgets.GridBox(
    children=buttons,
    layout=widgets.Layout(
        grid_template_columns="repeat(3, 85px)",
        grid_gap="7px",
        justify_content="center"
    )
)

restart = widgets.Button(
    description="New Game",
    button_style="info",
    layout=widgets.Layout(width="150px", height="40px")
)

# Update the screen
def update_screen():
    for i, button in enumerate(buttons):
        button.description = board[i] if board[i] else " "
        button.disabled = game_over or board[i] != ""

        if board[i] == "X":
            button.style.button_color = "#93c5fd"
        elif board[i] == "O":
            button.style.button_color = "#f9a8d4"
        else:
            button.style.button_color = "#ddd6fe"

    score.value = f"""
    <div style="text-align:center;font-size:17px;padding:12px">
    ✕ You: <b>{player_score}</b> &nbsp; | &nbsp;
    ○ ANN: <b>{computer_score}</b> &nbsp; | &nbsp;
    Draws: <b>{draw_score}</b>
    </div>
    """

# Finish a round
def finish_game():
    global game_over, player_score, computer_score, draw_score

    result = check_winner()

    if result == "X":
        game_over = True
        player_score += 1
        status.value = "<h3 style='text-align:center;color:green'>🎉 You Win!</h3>"

    elif result == "O":
        game_over = True
        computer_score += 1
        status.value = "<h3 style='text-align:center;color:#db2777'>🤖 ANN Wins!</h3>"

    elif result == "Draw":
        game_over = True
        draw_score += 1
        status.value = "<h3 style='text-align:center;color:#7c3aed'>🤝 Draw Game!</h3>"

    update_screen()
    return result is not None

# When player clicks a square
def play(index):
    if game_over or board[index] != "":
        return

    board[index] = "X"

    if finish_game():
        return

    move = computer_move()
    board[move] = "O"

    if not finish_game():
        status.value = "<h3 style='text-align:center;color:#7c3aed'>Your turn!</h3>"

    update_screen()

# Connect buttons
for i, button in enumerate(buttons):
    button.on_click(lambda b, i=i: play(i))

# Restart the game
def new_game(b):
    global board, game_over

    board = [""] * 9
    game_over = False
    status.value = "<h3 style='text-align:center;color:#7c3aed'>Your turn!</h3>"
    update_screen()

restart.on_click(new_game)

# Display game
update_screen()
display(status)
display(grid)
display(score)
display(restart)

display(HTML("""
<p style="text-align:center;color:#64748b">
Make three symbols in a row, column, or diagonal to win!
</p>
"""))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 61.8 MB/s eta 0:00:00
ANN trained successfully!


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


HTML(value="<h3 style='text-align:center;color:#7c3aed'>Your turn!</h3>")

GridBox(children=(Button(description=' ', layout=Layout(height='75px', width='85px'), style=ButtonStyle(button…

HTML(value='\n    <div style="text-align:center;font-size:17px;padding:12px">\n    ✕ You: <b>0</b> &nbsp; | &n…

Button(button_style='info', description='New Game', layout=Layout(height='40px', width='150px'), style=ButtonS…